In [2]:
import pandas as pd

data = pd.read_csv(r'C:\Users\acer\OneDrive\Documents\Fake-Review-Detection\data\fake_reviews.csv')

data.head()

,category,rating,label,text_
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...


In [3]:
data.shape


(40432, 4)

In [4]:
data.columns

Index(['category', 'rating', 'label', 'text_'], dtype='object')

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40432 entries, 0 to 40431
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   category  40432 non-null  object 
 1   rating    40432 non-null  float64
 2   label     40432 non-null  object 
 3   text_     40432 non-null  object 
dtypes: float64(1), object(3)
memory usage: 1.2+ MB


In [6]:
data['label'].value_counts()

label
CG    20216
OR    20216
Name: count, dtype: int64

In [7]:
data['label'] = data['label'].map({'CG':1, 'OR':0})

In [8]:
data['label'].value_counts()

label
1    20216
0    20216
Name: count, dtype: int64

In [9]:
data = data.rename(columns={'text_':'review'})

In [10]:
data.head()

,category,rating,label,review
0,Home_and_Kitchen_5,5.0,1,"Love this! Well made, sturdy, and very comfor..."
1,Home_and_Kitchen_5,5.0,1,"love it, a great upgrade from the original. I..."
2,Home_and_Kitchen_5,5.0,1,This pillow saved my back. I love the look and...
3,Home_and_Kitchen_5,1.0,1,"Missing information on how to use it, but it i..."
4,Home_and_Kitchen_5,5.0,1,Very nice set. Good quality. We have had the s...


In [11]:
data.isnull().sum()

category    0
rating      0
label       0
review      0
dtype: int64

In [12]:
data.dropna(inplace=True)

In [13]:
data.isnull().sum()

category    0
rating      0
label       0
review      0
dtype: int64

In [14]:
data = data[['review', 'label']]

In [15]:
data.head()

,review,label
0,"Love this! Well made, sturdy, and very comfor...",1
1,"love it, a great upgrade from the original. I...",1
2,This pillow saved my back. I love the look and...,1
3,"Missing information on how to use it, but it i...",1
4,Very nice set. Good quality. We have had the s...,1


In [16]:
data.shape

(40432, 2)

In [18]:
data.to_csv(r'C:\Users\acer\OneDrive\Documents\Fake-Review-Detection\data\cleaned_reviews.csv', index=False)

In [19]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\acer\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping corpora\stopwords.zip.


True

In [20]:
import re
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

In [21]:
def clean_text(text):
    text = text.lower()
    text = re.sub('[^a-z]', ' ', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

In [22]:
data['clean_review'] = data['review'].apply(clean_text)

In [23]:
data.head()

,review,label,clean_review
0,"Love this! Well made, sturdy, and very comfor...",1,love well made sturdy comfortable love pretty
1,"love it, a great upgrade from the original. I...",1,love great upgrade original mine couple years
2,This pillow saved my back. I love the look and...,1,pillow saved back love look feel pillow
3,"Missing information on how to use it, but it i...",1,missing information use great product price
4,Very nice set. Good quality. We have had the s...,1,nice set good quality set two months


In [24]:
data.to_csv(r'C:\Users\acer\OneDrive\Documents\Fake-Review-Detection\data\nlp_cleaned_reviews.csv', index=False)

In [25]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

In [26]:
X = data['clean_review']
y = data['label']

In [27]:
tfidf = TfidfVectorizer(max_features=5000)

X_tfidf = tfidf.fit_transform(X)

In [28]:
X_tfidf.shape

(40432, 5000)

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [30]:
X_train.shape

(32345, 5000)

In [31]:
X_test.shape

(8087, 5000)

In [32]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [33]:
# Import models and metrics
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Train Logistic Regression model
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

# Predict on test data
y_pred_lr = lr_model.predict(X_test)

# Accuracy
lr_accuracy = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", lr_accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8778286138246568

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.89      0.88      4071
           1       0.89      0.86      0.88      4016

    accuracy                           0.88      8087
   macro avg       0.88      0.88      0.88      8087
weighted avg       0.88      0.88      0.88      8087


Confusion Matrix:

[[3627  444]
 [ 544 3472]]


In [34]:
from sklearn.naive_bayes import MultinomialNB

# Train model
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

# Prediction
y_pred_nb = nb_model.predict(X_test)

# Accuracy
nb_accuracy = accuracy_score(y_test, y_pred_nb)
print("Naive Bayes Accuracy:", nb_accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_nb))

Naive Bayes Accuracy: 0.8547050822307407

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.82      0.85      4071
           1       0.83      0.89      0.86      4016

    accuracy                           0.85      8087
   macro avg       0.86      0.85      0.85      8087
weighted avg       0.86      0.85      0.85      8087


Confusion Matrix:

[[3354  717]
 [ 458 3558]]


In [35]:
from sklearn.ensemble import RandomForestClassifier

# Train model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

# Prediction
y_pred_rf = rf_model.predict(X_test)

# Accuracy
rf_accuracy = accuracy_score(y_test, y_pred_rf)
print("Random Forest Accuracy:", rf_accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_rf))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_rf))

Random Forest Accuracy: 0.8532212192407568

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.83      0.85      4071
           1       0.83      0.88      0.86      4016

    accuracy                           0.85      8087
   macro avg       0.85      0.85      0.85      8087
weighted avg       0.85      0.85      0.85      8087


Confusion Matrix:

[[3374  697]
 [ 490 3526]]


In [36]:
from sklearn.svm import LinearSVC

# Train model
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

# Prediction
y_pred_svm = svm_model.predict(X_test)

# Accuracy
svm_accuracy = accuracy_score(y_test, y_pred_svm)
print("SVM Accuracy:", svm_accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_svm))

SVM Accuracy: 0.8806726845554593

Classification Report:

              precision    recall  f1-score   support

           0       0.88      0.88      0.88      4071
           1       0.88      0.88      0.88      4016

    accuracy                           0.88      8087
   macro avg       0.88      0.88      0.88      8087
weighted avg       0.88      0.88      0.88      8087


Confusion Matrix:

[[3598  473]
 [ 492 3524]]


In [37]:
pip install xgboost

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.5/101.7 MB 2.2 MB/s eta 0:00:47
   ---------------------------------------- 1.0/101.7 MB 2.2 MB/s eta 0:00:46
    --------------------------------------- 1.6/101.7 MB 2.3 MB/s eta 0:00:44
    --------------------------------------- 2.1/101.7 MB 2.4 MB/s eta 0:00:43
   - -------------------------------------- 2.9/101.7 MB 2.6 MB/s eta 0:00:38
   - -------------------------------------- 3.1/101.7 MB 2.5 MB/s eta 0:00:39
   - -------------------------------------- 3.7/101.7 MB 2.3 MB/s eta 0:00:42
   - -------------------------------------- 3.9/101.7 MB 2.3 MB/s eta 0:00:44
   - -------------------------------------- 4.5/101.7 MB 2.2 MB/s eta 0:00:45
   - -------------------------------------- 5.0/101.7 MB 2.3 MB/s eta 0:00:43
   -- ------------------------------------- 6.3/101.7 MB 2.5 MB/s eta 0:00:38


In [38]:
from xgboost import XGBClassifier

# Train model
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

# Prediction
y_pred_xgb = xgb_model.predict(X_test)

# Accuracy
xgb_accuracy = accuracy_score(y_test, y_pred_xgb)
print("XGBoost Accuracy:", xgb_accuracy)

# Classification report
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_xgb))

# Confusion matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_xgb))

C:\Users\acer\anaconda3\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:56:40] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBoost Accuracy: 0.8438234203041919

Classification Report:

              precision    recall  f1-score   support

           0       0.82      0.88      0.85      4071
           1       0.87      0.81      0.84      4016

    accuracy                           0.84      8087
   macro avg       0.85      0.84      0.84      8087
weighted avg       0.85      0.84      0.84      8087


Confusion Matrix:

[[3564  507]
 [ 756 3260]]


In [39]:
tfidf2 = TfidfVectorizer(max_features=10000, ngram_range=(1,2))
X_tfidf2 = tfidf2.fit_transform(data['clean_review'])
y = data['label']

X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_tfidf2, y, test_size=0.2, random_state=42
)

In [40]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

lr = LogisticRegression(max_iter=1000)

lr.fit(X_train2, y_train2)

lr_pred = lr.predict(X_test2)

print("Logistic Regression Accuracy:", accuracy_score(y_test2, lr_pred))

print("\nClassification Report:\n")
print(classification_report(y_test2, lr_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, lr_pred))

Logistic Regression Accuracy: 0.8997155929269197

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.91      0.90      4071
           1       0.91      0.89      0.90      4016

    accuracy                           0.90      8087
   macro avg       0.90      0.90      0.90      8087
weighted avg       0.90      0.90      0.90      8087


Confusion Matrix:

[[3716  355]
 [ 456 3560]]


In [41]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

nb = MultinomialNB()

nb.fit(X_train2, y_train2)

nb_pred = nb.predict(X_test2)

print("Naive Bayes Accuracy:", accuracy_score(y_test2, nb_pred))

print("\nClassification Report:\n")
print(classification_report(y_test2, nb_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, nb_pred))

Naive Bayes Accuracy: 0.8759737850871769

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.89      0.88      4071
           1       0.88      0.86      0.87      4016

    accuracy                           0.88      8087
   macro avg       0.88      0.88      0.88      8087
weighted avg       0.88      0.88      0.88      8087


Confusion Matrix:

[[3613  458]
 [ 545 3471]]


In [42]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

svm = LinearSVC()

svm.fit(X_train2, y_train2)

svm_pred = svm.predict(X_test2)

print("SVM Accuracy:", accuracy_score(y_test2, svm_pred))

print("\nClassification Report:\n")
print(classification_report(y_test2, svm_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, svm_pred))

SVM Accuracy: 0.9105972548534685

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.92      0.91      4071
           1       0.91      0.91      0.91      4016

    accuracy                           0.91      8087
   macro avg       0.91      0.91      0.91      8087
weighted avg       0.91      0.91      0.91      8087


Confusion Matrix:

[[3729  342]
 [ 381 3635]]


In [43]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score
import numpy as np

In [44]:
# ==============================
# Logistic Regression Professional Model
# ==============================

lr_model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    n_jobs=-1,
    random_state=42
)

# Training
lr_model.fit(X_train2, y_train2)

# Prediction
lr_pred = lr_model.predict(X_test2)

# Accuracy
lr_accuracy = accuracy_score(y_test2, lr_pred)
print("Logistic Regression Accuracy:", lr_accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test2, lr_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, lr_pred))

# Cross Validation
cv_scores = cross_val_score(lr_model, X_tfidf2, y, cv=5)

print("\nCross Validation Scores:", cv_scores)
print("Average CV Score:", np.mean(cv_scores))

Logistic Regression Accuracy: 0.8997155929269197

Classification Report:

              precision    recall  f1-score   support

           0       0.89      0.91      0.90      4071
           1       0.91      0.89      0.90      4016

    accuracy                           0.90      8087
   macro avg       0.90      0.90      0.90      8087
weighted avg       0.90      0.90      0.90      8087


Confusion Matrix:

[[3716  355]
 [ 456 3560]]

Cross Validation Scores: [0.89402745 0.80202795 0.85246104 0.86581746 0.8362602 ]
Average CV Score: 0.8501188212862342


In [45]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score
import numpy as np

# ==============================
# Naive Bayes Professional Model
# ==============================

nb_model = MultinomialNB(alpha=0.1)

# Training
nb_model.fit(X_train2, y_train2)

# Prediction
nb_pred = nb_model.predict(X_test2)

# Accuracy
nb_accuracy = accuracy_score(y_test2, nb_pred)
print("Naive Bayes Accuracy:", nb_accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test2, nb_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, nb_pred))

# Cross Validation
cv_scores = cross_val_score(nb_model, X_tfidf2, y, cv=5)

print("\nCross Validation Scores:", cv_scores)
print("Average CV Score:", np.mean(cv_scores))

Naive Bayes Accuracy: 0.8804253740571287

Classification Report:

              precision    recall  f1-score   support

           0       0.87      0.90      0.88      4071
           1       0.89      0.87      0.88      4016

    accuracy                           0.88      8087
   macro avg       0.88      0.88      0.88      8087
weighted avg       0.88      0.88      0.88      8087


Confusion Matrix:

[[3644  427]
 [ 540 3476]]

Cross Validation Scores: [0.8598986  0.75528626 0.78889439 0.85320307 0.83774425]
Average CV Score: 0.819005313260831


In [46]:
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score
import numpy as np

# ==============================
# SVM Professional Model
# ==============================

svm_model = LinearSVC(
    C=1.0,
    max_iter=2000,
    random_state=42
)

# Training
svm_model.fit(X_train2, y_train2)

# Prediction
svm_pred = svm_model.predict(X_test2)

# Accuracy
svm_accuracy = accuracy_score(y_test2, svm_pred)
print("SVM Accuracy:", svm_accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test2, svm_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, svm_pred))

# Cross Validation
cv_scores = cross_val_score(svm_model, X_tfidf2, y, cv=5)

print("\nCross Validation Scores:", cv_scores)
print("Average CV Score:", np.mean(cv_scores))

SVM Accuracy: 0.9105972548534685

Classification Report:

              precision    recall  f1-score   support

           0       0.91      0.92      0.91      4071
           1       0.91      0.91      0.91      4016

    accuracy                           0.91      8087
   macro avg       0.91      0.91      0.91      8087
weighted avg       0.91      0.91      0.91      8087


Confusion Matrix:

[[3729  342]
 [ 381 3635]]

Cross Validation Scores: [0.89489304 0.80919995 0.84355676 0.87583478 0.8350235 ]
Average CV Score: 0.851701605417048


In [47]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# ==============================
# Random Forest Professional Model
# ==============================

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)

# Training
rf_model.fit(X_train2, y_train2)

# Prediction
rf_pred = rf_model.predict(X_test2)

# Accuracy
rf_accuracy = accuracy_score(y_test2, rf_pred)
print("Random Forest Accuracy:", rf_accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test2, rf_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, rf_pred))

# Cross Validation
cv_scores = cross_val_score(rf_model, X_tfidf2, y, cv=5)

print("\nCross Validation Scores:", cv_scores)
print("Average CV Score:", np.mean(cv_scores))

Random Forest Accuracy: 0.8647211574131322

Classification Report:

              precision    recall  f1-score   support

           0       0.86      0.88      0.87      4071
           1       0.87      0.85      0.86      4016

    accuracy                           0.86      8087
   macro avg       0.87      0.86      0.86      8087
weighted avg       0.86      0.86      0.86      8087


Confusion Matrix:

[[3578  493]
 [ 601 3415]]

Cross Validation Scores: [0.85928033 0.78150117 0.82859263 0.82043037 0.81152609]
Average CV Score: 0.8202661196759541


In [48]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score
import numpy as np

# ==============================
# XGBoost Professional Model
# ==============================

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

# Training
xgb_model.fit(X_train2, y_train2)

# Prediction
xgb_pred = xgb_model.predict(X_test2)

# Accuracy
xgb_accuracy = accuracy_score(y_test2, xgb_pred)
print("XGBoost Accuracy:", xgb_accuracy)

# Classification Report
print("\nClassification Report:\n")
print(classification_report(y_test2, xgb_pred))

# Confusion Matrix
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test2, xgb_pred))

# Cross Validation
cv_scores = cross_val_score(xgb_model, X_tfidf2, y, cv=5)

print("\nCross Validation Scores:", cv_scores)
print("Average CV Score:", np.mean(cv_scores))

XGBoost Accuracy: 0.8548287374799061

Classification Report:

              precision    recall  f1-score   support

           0       0.83      0.89      0.86      4071
           1       0.88      0.82      0.85      4016

    accuracy                           0.85      8087
   macro avg       0.86      0.85      0.85      8087
weighted avg       0.86      0.85      0.85      8087


Confusion Matrix:

[[3622  449]
 [ 725 3291]]

Cross Validation Scores: [0.85890936 0.77136144 0.82673757 0.79408855 0.80719763]
Average CV Score: 0.8116589099479349


In [49]:
import pickle

# save svm model
pickle.dump(svm_model, open('svm_model.pkl', 'wb'))

# save tfidf
pickle.dump(tfidf2, open('tfidf_vectorizer.pkl', 'wb'))

In [50]:
import os
os.listdir()

['.anaconda',
 '.conda',
 '.continuum',
 '.ipynb_checkpoints',
 '.ipython',
 '.jupyter',
 '.matplotlib',
 '.ms-ad',
 '.vscode',
 '0and1pro.ipynb',
 'anaconda3',
 'AppData',
 'Application Data',
 'classified_images1',
 'Contacts',
 'Cookies',
 'data_preprocessing.ipynb',
 'Downloads',
 'extracted_frames',
 'extracted_frames01',
 'extracted_frames02',
 'extracted_frames03',
 'extracted_frames1',
 'extracted_frames10',
 'extracted_frames11',
 'extracted_frames12',
 'extracted_frames13',
 'extracted_frames14',
 'extracted_frames15',
 'extracted_frames16',
 'extracted_frames17',
 'extracted_frames18',
 'extracted_frames19',
 'extracted_frames2',
 'extracted_frames20',
 'extracted_frames21',
 'extracted_frames22',
 'extracted_frames23',
 'extracted_frames24',
 'extracted_frames25',
 'extracted_frames26',
 'extracted_frames27',
 'extracted_frames28',
 'extracted_frames29',
 'extracted_frames30',
 'extracted_frames31',
 'extracted_frames32',
 'extracted_frames33',
 'extracted_frames34',
 'extr

In [51]:
# ==============================
# Load Model and Predict
# ==============================

import pickle

model = pickle.load(open('svm_model.pkl', 'rb'))
tfidf = pickle.load(open('tfidf_vectorizer.pkl', 'rb'))

def predict_review(review):
    
    review_tfidf = tfidf.transform([review])
    prediction = model.predict(review_tfidf)[0]

    if prediction == 1:
        return "Fake Review"
    else:
        return "Genuine Review"


# Testing
print(predict_review("This product is amazing and very useful"))
print(predict_review("Worst product ever, totally waste of money"))
print(predict_review("I really love this item and quality is good"))

Genuine Review
Genuine Review
Fake Review
